# System 4: RoBERTa-base on a free Colab GPU

Fine-tunes RoBERTa-base for anti-Muslim hate speech detection and scores it
through the same harness used by every other system in the project, so the
numbers drop straight into the cross-model comparison.

**Before running:** Runtime -> Change runtime type -> T4 GPU.

Three seeds at 3 epochs takes roughly 25-40 minutes on a T4. Drop to
`--seeds 42` first if you only want to check it runs.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Get the repo

Replace the URL with your own remote. If you would rather not push the repo
anywhere, zip the project folder, upload it with the Files pane, and
`!unzip -q hate-speech-nlp-evolution.zip` instead.

In [ ]:
REPO_URL = "https://github.com/<your-username>/hate-speech-nlp-evolution.git"

import os

if not os.path.exists("hate-speech-nlp-evolution"):
    !git clone -q $REPO_URL
%cd hate-speech-nlp-evolution
!ls

## 3. Dependencies

Colab already ships torch built against the right CUDA, so it is left alone
here -- installing a pinned torch on top is the usual way to break a Colab
runtime. Only what is missing gets installed.

In [ ]:
!pip install -q transformers accelerate

import torch

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 4. Download the corpora and build the splits

This verifies the thirteen pilot counts and refuses to continue if any of
them moved. The split is seeded, so the one built here is identical to the
one used locally and the results stay comparable.

In [ ]:
!python -m scripts.prepare_data

## 5. Fine-tune

One run per seed. Each is scored by the shared harness and written to
`results/runs/`; the mean +/- std aggregate goes to `results/roberta_base.json`.

If the T4 runs out of memory, drop `--batch-size` to 8.

In [ ]:
!python -m scripts.run_transformer \
    --model roberta-base \
    --seeds 42 43 44 \
    --epochs 3 \
    --batch-size 16 \
    --name roberta_base

## 6. Comparison tables

Only shows the systems whose results JSON is present. If you ran the
lexicon and the SVM locally, upload their `results/*.json` first so all
three appear.

In [ ]:
!python -m scripts.make_comparison_table

## 7. Save the results back

Colab discards the filesystem when the runtime ends. Download the results
before closing the tab.

In [ ]:
from google.colab import files

!zip -qr results.zip results/
files.download("results.zip")